# HW8 (20')

<font size='4'>

For this assignment, it is a combination of jupyter notebook assignment and python scripts.

For Q1, please upload your outputs including codes and graphics to your own GitHub repository. <br> You will need to disclose your GitHub repository below.

For Q2, please submit this jupyter notebook as an HTML or PDF file.

First of all, print your name (First and Last) below.

In [1]:
print('Tej Shah')

Tej Shah


## 0. Import relevant packages

In [4]:
import os
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.formula.api as smf

## Q1. Convert your HW7 to python scripts. (10')

<font size='4'>

- Under your working directory, there should be a folder called `self_py_fun`.
- Create a new python file called `HW8Fun.py` and move previously defined functions `produce_trun_mean_cov()`, `plot_trunc_mean()`, and `plot_trunc_cov()` to that file. Make sure you import proper packages.
- Create another main file `HW8_main.py`.
- Import relevant packages, modules, and/or function.
- Copy the global variables and call your functions inside `HW8_main.py`.
- A major difference compared to HW7 is that you are asked to save those figures to your local working environment.
    - Create a new directory `K114` under your current working directory.
    - For mean functions, please save it as a `Mean.png` output using `plt.savefig()` function.
    - The changes should be made within `HW8Fun.py` rather than `HW8_main.py`.
    - For covariance matrices, please save them as `Covariance_Target.png`, `Covariance_Non-Target.png`, and `Covariance_All.png` outputs using the same function above.
    - To summarize, there should be **four** figures under `K114` folder.
- Upload your entire work to your GitHub repository via push button.

In [90]:
# Provide your GitHub repository link below in the Markdown chunk. Remember to make it public and make the link clickable.
# Do not include sensitive information in your GitHub repository.

https://github.com/TejShah01/BIOS-584

## Q2. A real-world data anlaysis using `Pandas` and `Scipy` (10')

<font size='4'>

- Back to the `PTSD dataset.xlsx`, let's import the dataset and name it `ptsd_df`. (no point since everyone has done it a couple of times before.)

In [5]:
# Write your own code
file_path = "/Users/Tej/Documents/GitHub/BIOS-584/data/PTSD dataset.xlsx"
ptsd_df = pd.read_excel(file_path, sheet_name="main_dataset")

### Q2.1. Univariate comparison (3')

<font size='4'> 
    
- Suppose that we would like to examine the utility/effect of an intervention program for patients with PTSD.
- We measure PCL5 scores at completion (`pcl5week_score.completion`) and PCL5 score at 3-month follow-up (`pcl5month_score.3_month_follow_up`). Let's assume the first score is pre-intervention and the second score is post-intervention.
- Report the summary statistics for each variable including mean, std, median, Q1, and Q3.
- Note that each patient will receive such two PCL5 scores. Use a appropriate statistical test to perform the univariate comparison. Report the outputing statistic and p-value.
- Before you run the statistic test, determine the data type and check the missingness of two columns. In particular, report the number of NA values for each variable.

In [16]:
# Write your own code
col_pre  = 'pcl5week_score.completion'         
col_post = 'pcl5month_score.3_month_follow_up'

print(ptsd_df[[col_pre, col_post]].dtypes)
print(ptsd_df[[col_pre, col_post]].isna().sum())

df_pair = ptsd_df[[col_pre, col_post]].dropna()

def summarize(series):
    return pd.Series({
        'Mean': round(series.mean(), 2),
        'SD': round(series.std(), 2),
        'Median': round(series.median(), 2),
        'Q1': round(series.quantile(0.25), 2),
        'Q3': round(series.quantile(0.75), 2),
        'n': series.shape[0]
    })

summary = pd.concat([summarize(df_pair[col_pre]), summarize(df_pair[col_post])], axis=1)
summary.columns = ['PCL5_Completion (Pre)', 'PCL5_3Month (Post)']

display(summary)

stat, p_val = stats.ttest_rel(df_pair[col_post], df_pair[col_pre])

print("Outputing Statistic: ", stat)
print("P-Value: ", p_val)

pcl5week_score.completion            float64
pcl5month_score.3_month_follow_up    float64
dtype: object
pcl5week_score.completion             27
pcl5month_score.3_month_follow_up    251
dtype: int64


,PCL5_Completion (Pre),PCL5_3Month (Post)
Mean,27.21,32.03
SD,19.18,18.98
Median,22.00,31.00
Q1,12.75,16.00
Q3,41.00,46.00
n,224.00,224.00


Outputing Statistic:  4.709238006778945
P-Value:  4.370038747681667e-06


### Q2.2. Multiple Linear Regression (7')

<font size='4'>

- Select columns specified in the following code chunk and create a subset dataset named `ptsd_sub_df`.
- Fit a linear regression to examine the association between `caps_intake` (continuous outcome) and the remaining covariates (as predictors) using `ptsd_sub_df`.
    - Note that all covariates ending with `_code` are categorical variables.
- Use the instruction here to write the formula for linear regression in Python.
    - https://www.statsmodels.org/stable/example_formulas.html
- Report the output page including R2, adjusted R2, and parameter estimates, SE, 95% confidence intervals, and p-values.
- Provide a brief interpretation for all significant predictors (p<0.05) excluding the intercept.
- Relevant label information includes:
    - `employment_code`: 1: Employed, 2: Unemployed, 3: Retired, 4: Disabled/Unable to work, 5: Student, 6: Other.
    - `rank_code`: 1. Enlisted, 2: Officer, 3: Other

In [6]:
# The following column names are used for linear regression.
# Do not delete.
relevant_col_names = ['caps_intake', 'age_iop', 'gender_code', 'sexualorient_code', 'race_code', 'ethnicity_code', 
                      'education_code', 'employment_code',
                      'rank_code', 'branch_code', 'mdd_code', 'ctq_total_score', 'sexual_trauma', 'sud_code']

In [7]:
# Write your own code
ptsd_sub_df = ptsd_df[relevant_col_names].dropna()

cat_vars = [col for col in relevant_col_names if col.endswith('_code')]
cont_vars = [col for col in relevant_col_names if col not in cat_vars and col != 'caps_intake']

formula_terms = cont_vars + [f"C({v})" for v in cat_vars]
formula = "caps_intake ~ " + " + ".join(formula_terms)

model = smf.ols(formula=formula, data=ptsd_sub_df).fit()

results = pd.DataFrame({
    'Coefficient': model.params,
    'SE': model.bse,
    't': model.tvalues,
    'p_value': model.pvalues,
    'CI_lower': model.conf_int()[0],
    'CI_upper': model.conf_int()[1]
}).round(4)

print(f"R-squared: {model.rsquared:.3f}")
print(f"Adjusted R-squared: {model.rsquared_adj:.3f}\n")

display(results)


R-squared: 0.114
Adjusted R-squared: 0.051



,Coefficient,SE,t,p_value,CI_lower,CI_upper
Intercept,39.2629,2.7215,14.4270,0.0000,33.9134,44.6125
C(gender_code)[T.2],-1.5582,1.1307,-1.3780,0.1689,-3.7808,0.6645
C(sexualorient_code)[T.2],1.8247,1.9934,0.9154,0.3605,-2.0936,5.7430
C(sexualorient_code)[T.3],1.8012,2.0979,0.8586,0.3911,-2.3226,5.9250
C(race_code)[T.2],0.5325,1.0442,0.5100,0.6103,-1.5199,2.5850
C(race_code)[T.3],-3.2892,2.7936,-1.1774,0.2397,-8.7805,2.2022
C(race_code)[T.4],3.2360,5.1270,0.6312,0.5283,-6.8420,13.3141
C(race_code)[T.5],-2.8901,2.7545,-1.0492,0.2947,-8.3045,2.5243
C(race_code)[T.6],-3.5467,1.9731,-1.7975,0.0730,-7.4251,0.3318
C(ethnicity_code)[T.2],-2.8840,1.5673,-1.8401,0.0665,-5.9648,0.1968


# Write your interpretations below:


As the multiple linear regression table shows, employment code #4, i.e., people who are disabled/are unable to work and a higher score on the CTQ survey, which tests for Childhood Trauma, have a p-value less than 0.05 and a positive coefficient (parameter estimate) when compared to the intercept of CAPS Intake scores, meaning that these variables are positively associated and may be predictors of poor mental health as determined by the CAPS intake scores.

On the other hand, rank code #2, i.e., officers have a negative coefficient score and a p-value less than 0.05, which means it is negatively correlated with CAPS intake score and may have a protective effect on mental health. 

All other variables with p-values greater than 0.05 do not have a statistically significant relationship with CAPS intake scores.